# 10 · Segmentacion no supervisada

> **Objetivo:** aplicar k-means y DBSCAN sobre el mismo preprocesador
> del caso supervisado, y comparar los segmentos resultantes con las
> acciones de marketing que U4 presenta.

## Que vamos a hacer

1. Reutilizar `construir_preprocesador` para que la matriz sea identica
   a la del cuaderno 09; sin esto, los clusters no son comparables.
2. Elegir `k` con silhouette; comparar con DBSCAN.
3. Perfilar los segmentos (medias por cluster) y asignarles nombres
   accionables.

> Este cuaderno consolida k-means, DBSCAN y la comparacion de
> modelos en un solo flujo polars; sustituye los tres cuadernos de
> clustering del flujo pandas original de la Unidad 2.

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from segmentacion.config import FECHA_CORTE_A, HORIZONTE
from segmentacion.data.cargar import leer_parquet
from segmentacion.data.limpieza import preparar_ventas
from segmentacion.features.cohortes import cohorte
from segmentacion.models.clustering import (
    evaluate_clustering,
    find_optimal_k,
    fit_dbscan,
    fit_kmeans,
)
from segmentacion.models.profiling import build_segment_profiles
from segmentacion.pipelines.preprocesamiento import CAT, NUM_LOG, NUM_STD, construir_preprocesador

ventas, canc = preparar_ventas(leer_parquet(PROJECT_ROOT / "data" / "raw" / "online_retail_II.parquet"), log=False)
A = cohorte(ventas, canc, FECHA_CORTE_A, horizonte=HORIZONTE)
cols = NUM_LOG + NUM_STD + CAT
XA = A.select(cols).to_pandas()

pre = construir_preprocesador()
X_prep = pre.fit_transform(XA)
print(f"matriz lista para clustering: {X_prep.shape}")

matriz lista para clustering: (4933, 8)


## 1. k-means con silhouette para elegir k

In [3]:

tabla_k = find_optimal_k(X_prep, k_range=range(2, 8), random_state=42)
tabla_k.round(3)

[23:17:58] INFO    segmentacion.models.clustering | K-Means k=2 entrenado | inertia=19963.55 | iter=9


[23:17:58] INFO    segmentacion.models.clustering | K-Means k=3 entrenado | inertia=16312.70 | iter=9


[23:17:59] INFO    segmentacion.models.clustering | K-Means k=4 entrenado | inertia=13715.00 | iter=28


[23:17:59] INFO    segmentacion.models.clustering | K-Means k=5 entrenado | inertia=11776.29 | iter=28


[23:17:59] INFO    segmentacion.models.clustering | K-Means k=6 entrenado | inertia=10950.03 | iter=15


[23:17:59] INFO    segmentacion.models.clustering | K-Means k=7 entrenado | inertia=10233.18 | iter=31


,inertia,silhouette,davies_bouldin,calinski_harabasz
k,,,,
2,19963.547,0.307,1.241,2567.476
3,16312.697,0.288,1.356,2122.411
4,13715.001,0.299,1.167,1993.783
5,11776.286,0.258,1.195,1943.983
6,10950.031,0.234,1.327,1746.555
7,10233.178,0.208,1.378,1614.613


In [4]:
k_optimo = int(tabla_k["silhouette"].idxmax())
print(f"k optimo por silhouette: {k_optimo}")
kmeans = fit_kmeans(X_prep, n_clusters=k_optimo, random_state=42)
etiquetas_km = kmeans.labels_

k optimo por silhouette: 2
[23:17:59] INFO    segmentacion.models.clustering | K-Means k=2 entrenado | inertia=19963.55 | iter=9


## 2. DBSCAN como contraste

In [5]:
dbscan = fit_dbscan(X_prep, eps=0.5, min_samples=10)
print(evaluate_clustering(X_prep, dbscan.labels_))

[23:17:59] INFO    segmentacion.models.clustering | DBSCAN eps=0.5 min_samples=10 | clusters=13 | ruido=2262 (45.9%)


{'silhouette': -0.14550646891332905, 'davies_bouldin': 1.1954478066679093, 'calinski_harabasz': 183.9558719963082, 'n_clusters': 13, 'n_noise': 2262}


DBSCAN produce clusters desbalanceados y mucho ruido (etiqueta
`-1`); la silueta cae. En este dataset la densidad **no** separa bien
porque la mayoria de los clientes vive en una nube unica alrededor del
origen. k-means con `k = 4` es la eleccion razonable; DBSCAN sirve para
detectar outliers (mayoristas, fraudes).

## 3. Perfilar y nombrar los segmentos

In [6]:
# `build_segment_profiles` promedia por cluster; excluimos `pais` (string).
perfiles = build_segment_profiles(XA.drop(columns=['pais']), etiquetas_km)
perfiles.round(2)

,n_customers,monetario,frecuencia,productos_distintos,recencia,antiguedad,tasa_devolucion,share
cluster,,,,,,,,
0,2117,4932.35,9.96,128.2,89.50,446.81,0.25,0.43
1,2816,486.58,1.69,26.5,228.25,289.81,0.18,0.57


Convertimos los centroides en nombres interpretables. La
heuristica usa cuartiles de `recencia`, `frecuencia` y `monetario`.

## 4. Fichas de accion

El equipo de U4 recibe estos perfiles y la ficha del modelo supervisado
(`models/recompra_v1/metadatos.json`). La app Streamlit presenta las
dos vistas lado a lado: segmentos y probabilidad de recompra.

## Preguntas de reflexion

1. ¿Por que la silueta de DBSCAN cae tanto? ¿Es un problema del
   dataset o de los hiperparametros?
2. ¿Como combinarias los segmentos de k-means con la probabilidad del
   bosque para una campaña? Piensa en 2 x 2: segmento x probabilidad.
3. Si DBSCAN marca 50 clientes como ruido, ¿los pasas a marketing
   como "atipicos a investigar" o los excluyes de las campañas?